In [1]:
import torch
from torch import nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Memory: 6.44 GB


In [2]:
data_dir = "../data/chest_xray"
train_path = os.path.join(data_dir, "train")
test_path = os.path.join(data_dir, "test")

# Defining the transforms for the datasets
train_transforms = transforms.Compose([
    transforms.Resize((224, 244)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 244)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load FULL original training data
full_train_dataset = datasets.ImageFolder(train_path, transform=train_transforms)

# Split 80-20 for training and validation
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Test dataset remains original (624 images)
test_dataset = datasets.ImageFolder(test_path, transform=val_transforms)

# Preparing the dataloaders for our datasets
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=16, num_workers=2)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=16, num_workers=2)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=16, num_workers=2)

print(f"Train size: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train size: 4172 | Val: 1044 | Test: 624


In [3]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Step 1: Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Step 2: Unfreeze BatchNorm layers everywhere
for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        for param in module.parameters():
            param.requires_grad = True

# Step 3: Unfreeze the LAST ResNet block (layer4)
# ResNet18 structure: conv1, bn1, relu, maxpool, layer1, layer2, layer3, layer4, avgpool, fc
for param in model.layer4.parameters():
    param.requires_grad = True

# Step 4: Replace FC head
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 2)
)

In [4]:
model = model.to(device)

# Simple approach: collect all trainable params
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(trainable_params, lr=0.001, weight_decay=1e-4)

criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        _, predicted = torch.max(pred, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

        epoch_loss = running_loss / total
        epoch_acc = correct / total
        return epoch_loss, epoch_acc
    
def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

num_epochs = 25
best_val_acc = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_dataloader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, val_dataloader, criterion, device)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '../backend/models/resnet18_pneumonia_best.pth')
        print(f"** Saved best model with val acc: {val_acc:.4f} **")
    
    print("-" * 40)

Epoch 1/25
Train Loss: 0.6624 | Train Acc: 0.6250
Val Loss: 0.5505 | Val Acc: 0.7261
** Saved best model with val acc: 0.7261 **
----------------------------------------
Epoch 2/25
Train Loss: 0.2935 | Train Acc: 0.8750
Val Loss: 0.9639 | Val Acc: 0.7261
----------------------------------------
Epoch 3/25
Train Loss: 0.6345 | Train Acc: 0.7500
Val Loss: 0.9726 | Val Acc: 0.7289
** Saved best model with val acc: 0.7289 **
----------------------------------------
Epoch 4/25
Train Loss: 0.2532 | Train Acc: 0.8750
Val Loss: 0.9259 | Val Acc: 0.7739
** Saved best model with val acc: 0.7739 **
----------------------------------------
Epoch 5/25
Train Loss: 0.3038 | Train Acc: 0.8125
Val Loss: 0.6948 | Val Acc: 0.8372
** Saved best model with val acc: 0.8372 **
----------------------------------------
Epoch 6/25
Train Loss: 0.1683 | Train Acc: 0.9375
Val Loss: 0.5654 | Val Acc: 0.8640
** Saved best model with val acc: 0.8640 **
----------------------------------------
Epoch 7/25
Train Loss: 0

In [ ]:
import torch

checkpoint = torch.load('../backend/models/resnet18_pneumonia_best.pth', map_location=device)
model.load_state_dict(checkpoint)
test_loss, test_acc = validate_epoch(model, test_dataloader, criterion, device)
print(f"Final Test Accuracy: {test_acc:.4f}")  # Should be ~85-90%

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13648\3931284602.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('../backend/models/resnet18_pneumon

Final Test Accuracy: 0.8269


In [5]:
# Save complete package for FastAPI
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': ['NORMAL', 'PNEUMONIA'],
    'input_size': 224,
    'normalization': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225]
    },
    'architecture': 'resnet18_custom'
}, '../backend/models/pneumonia_model_full.pt')

print("✅ Model saved for FastAPI deployment!")

✅ Model saved for FastAPI deployment!


In [6]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['NORMAL', 'PNEUMONIA']))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.50      0.82      0.62       234
   PNEUMONIA       0.82      0.51      0.63       390

    accuracy                           0.63       624
   macro avg       0.66      0.66      0.63       624
weighted avg       0.70      0.63      0.63       624

Confusion Matrix:
[[191  43]
 [190 200]]


In [7]:
from sklearn.metrics import precision_recall_curve

# Get validation probabilities
model.eval()
val_probs = []
val_labels = []

with torch.no_grad():
    for images, labels in val_dataloader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        val_probs.extend(probs[:, 1].cpu().numpy())  # probability of PNEUMONIA
        val_labels.extend(labels.cpu().numpy())

val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

# Compute precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(val_labels, val_probs)

# Find threshold that gives high precision (fewer false positives) while maintaining decent recall
# Let's target precision >= 0.9
target_precision = 0.9
idx = np.where(precisions[:-1] >= target_precision)[0]
if len(idx) > 0:
    optimal_threshold = thresholds[idx[0]]
    print(f"Optimal threshold for precision >= {target_precision}: {optimal_threshold:.4f}")
else:
    # Fallback: use threshold at max F1
    from sklearn.metrics import f1_score
    f1_scores = [f1_score(val_labels, val_probs >= t) for t in thresholds]
    optimal_threshold = thresholds[np.argmax(f1_scores)]
    print(f"Optimal threshold by F1: {optimal_threshold:.4f}")

Optimal threshold for precision >= 0.9: 0.6149


In [8]:
test_probs = []
test_labels = []
with torch.no_grad():
    for images, labels in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        test_probs.extend(probs[:, 1].cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

new_preds = (np.array(test_probs) >= optimal_threshold).astype(int)
print("With threshold", optimal_threshold)
print(classification_report(test_labels, new_preds, target_names=['NORMAL', 'PNEUMONIA']))

With threshold 0.6149067
              precision    recall  f1-score   support

      NORMAL       0.38      1.00      0.55       234
   PNEUMONIA       0.00      0.00      0.00       390

    accuracy                           0.38       624
   macro avg       0.19      0.50      0.27       624
weighted avg       0.14      0.38      0.20       624



c:\Users\Lenovo\anaconda3\envs\deeplearning\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\anaconda3\envs\deeplearning\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\anaconda3\envs\deeplearning\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [9]:
# Only for training - oversample Normal class
from torch.utils.data import WeightedRandomSampler

# Get labels of training dataset
train_labels = [label for _, label in train_dataset]
class_counts = np.bincount(train_labels)
class_weights = 1. / class_counts
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=2)

In [10]:
from sklearn.metrics import accuracy_score, recall_score, f1_score


def run_epoch_with_metrics(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_recall = recall_score(all_labels, all_preds, average='binary', pos_label=1)
    epoch_f1 = f1_score(all_labels, all_preds, average='binary', pos_label=1)

    return epoch_loss, epoch_acc, epoch_recall, epoch_f1


# Use weighted sampler loader if it exists, otherwise fall back to the original train dataloader.
active_train_loader = train_loader if 'train_loader' in globals() else train_dataloader

num_epochs = 10
best_val_f1 = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc, train_recall, train_f1 = run_epoch_with_metrics(
        model, active_train_loader, criterion, device, optimizer=optimizer
    )

    val_loss, val_acc, val_recall, val_f1 = run_epoch_with_metrics(
        model, val_dataloader, criterion, device, optimizer=None
    )

    scheduler.step(val_loss)

    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(
        f"Train -> Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | "
        f"Recall: {train_recall:.4f} | F1: {train_f1:.4f}"
    )
    print(
        f"Val   -> Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | "
        f"Recall: {val_recall:.4f} | F1: {val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), '../backend/models/resnet18_pneumonia_best_finetuned.pth')
        print(f"Saved best fine-tuned model (Val F1: {best_val_f1:.4f})")

    print('-' * 50)

print(f"Best validation F1: {best_val_f1:.4f}")

Epoch 1/10
Train -> Loss: 0.1398 | Acc: 0.9537 | Recall: 0.9485 | F1: 0.9529
Val   -> Loss: 0.0628 | Acc: 0.9770 | Recall: 0.9746 | F1: 0.9846
Saved best fine-tuned model (Val F1: 0.9846)
--------------------------------------------------
Epoch 2/10
Train -> Loss: 0.0794 | Acc: 0.9734 | Recall: 0.9693 | F1: 0.9733
Val   -> Loss: 0.0806 | Acc: 0.9655 | Recall: 0.9581 | F1: 0.9767
--------------------------------------------------
Epoch 3/10
Train -> Loss: 0.0700 | Acc: 0.9775 | Recall: 0.9763 | F1: 0.9777
Val   -> Loss: 0.0682 | Acc: 0.9741 | Recall: 0.9721 | F1: 0.9827
--------------------------------------------------
Epoch 4/10
Train -> Loss: 0.0609 | Acc: 0.9815 | Recall: 0.9783 | F1: 0.9814
Val   -> Loss: 0.0664 | Acc: 0.9732 | Recall: 0.9683 | F1: 0.9820
--------------------------------------------------
Epoch 5/10
Train -> Loss: 0.0458 | Acc: 0.9839 | Recall: 0.9819 | F1: 0.9836
Val   -> Loss: 0.0640 | Acc: 0.9770 | Recall: 0.9708 | F1: 0.9846
------------------------------------